In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ems.db import load_env, connect

load_env()

In [2]:
# 대상 계량기 정의
COOLING_ELECTRIC = ['H1.Z16', 'H1.Z11', 'H1.Z12', 'H1.Z24', 'H1.Z25']
COOLING_THERMAL  = ['V.K21', 'H1.K11', 'H1.K12', 'H1.K14', 'H1.K15', 'H1.K16', 'H2.K21']
HEATING_ELECTRIC = ['H1.Z20', 'H1.ZE20']
HEATING_THERMAL  = ['H1.W11', 'H1.W12']

ALL_METERS = COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL

# 조회 기간: 2023년 50주차 (12월 11일~12월 17일)
START = '2023-12-11'
END   = '2023-12-18'

print(f'전체 계량기 수: {len(ALL_METERS)}')
print(f'조회 기간: {START} ~ {END}')

전체 계량기 수: 16
조회 기간: 2023-12-11 ~ 2023-12-18


In [3]:
def fetch_meter_data(meter_urn: str) -> pd.DataFrame:
    sql = """
        SELECT
            ts,
            measurement,
            value
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND ts >= %s
          AND ts <  %s
        ORDER BY ts, measurement
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, START, END))

    if df.empty:
        return pd.DataFrame()

    df = df.pivot(index='ts', columns='measurement', values='value')
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Europe/Berlin')
    df.index.name = 'timestamp'
    return df

In [4]:
# Step 1: CSV 저장
import os

save_dir_csv = ROOT / 'outputs/tables/stl_eda/daily'
os.makedirs(save_dir_csv, exist_ok=True)

WEEKDAY_MAP = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}

for meter in ALL_METERS:
    df = fetch_meter_data(meter)
    if df.empty:
        print(f'{meter} 데이터 없음')
        continue

    df['hour']    = df.index.hour
    df['weekday'] = df.index.weekday
    df['day_label'] = df.index.strftime('%m/%d') + '(' + df.index.weekday.map(WEEKDAY_MAP) + ')'

    # 시간별 평균 (날짜별 분리)
    hourly_mean = df.groupby(['day_label', 'hour']).mean(numeric_only=True)
    hourly_mean.to_csv(save_dir_csv / f'{meter}_daily_mean.csv')
    print(f'{meter} 저장 완료')

/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.Z16 저장 완료


/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.Z11 저장 완료
H1.Z12 저장 완료


/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))
/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


...

/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))
/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.Z20 저장 완료
H1.ZE20 저장 완료


/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))
/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.W11 저장 완료
H1.W12 저장 완료


/tmp/ipykernel_63999/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


In [5]:
# Step 2: PNG 저장
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

def plot_daily_trend(meter_urn: str, df: pd.DataFrame, title_suffix: str = '') -> go.Figure:
    df = df.reset_index()
    measurements = [c for c in df.columns if c not in ['day_label', 'hour', 'weekday']]
    n_meas = len(measurements)
    ncols = min(4, n_meas)
    nrows = (n_meas + ncols - 1) // ncols

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=measurements,
        shared_xaxes=False,
    )

    days = sorted(df['day_label'].unique())

    for idx, meas in enumerate(measurements):
        row = idx // ncols + 1
        col = idx % ncols + 1

        for d_idx, day in enumerate(days):
            day_data = df[df['day_label'] == day].sort_values('hour')
            fig.add_trace(
                go.Scatter(
                    x=day_data['hour'],
                    y=day_data[meas],
                    name=day,
                    legendgroup=day,
                    showlegend=(idx == 0),
                    line=dict(color=COLORS[d_idx % len(COLORS)]),
                    mode='lines+markers',
                    marker=dict(size=4),
                ),
                row=row, col=col
            )

        fig.update_xaxes(
            tickvals=list(range(0, 24, 2)),
            ticktext=[f'{h}시' for h in range(0, 24, 2)],
            row=row, col=col
        )

    fig.update_layout(
        title=f'{meter_urn} 일간 트렌드 (시간별, 2023년 12월 50주차) {title_suffix}',
        height=max(300, nrows * 250),
        width=1400,
        legend=dict(title='날짜'),
        template='plotly_white',
    )
    return fig


save_dir_png = ROOT / 'outputs/figures/stl_eda/daily'
os.makedirs(save_dir_png, exist_ok=True)

for meter in ALL_METERS:
    csv_path = save_dir_csv / f'{meter}_daily_mean.csv'
    if not csv_path.exists():
        print(f'{meter} CSV 없음')
        continue

    df = pd.read_csv(csv_path)
    fig = plot_daily_trend(meter, df, '')
    fig.write_image(
        str(save_dir_png / f'{meter}_daily_trend.png'),
        width=1400, height=800
    )
    print(f'{meter} 저장 완료')

H1.Z16 저장 완료
H1.Z11 저장 완료
H1.Z12 저장 완료
H1.Z24 저장 완료
H1.Z25 저장 완료
V.K21 저장 완료
H1.K11 저장 완료
H1.K12 저장 완료
H1.K14 저장 완료
H1.K15 저장 완료
H1.K16 저장 완료
H2.K21 저장 완료
H1.Z20 저장 완료
H1.ZE20 저장 완료
H1.W11 저장 완료
H1.W12 저장 완료
